Pattern 1: Linear Recursion — Process and Recurse

In [1]:
def sum_array(arr: list[int], index: int = 0) -> int:
    """
    Recurrence: T(n) = T(n-1) + O(1)
    Solution:   T(n) = O(n)
    
    Process current element, recurse on the rest.
    """
    # Base case: past the end of array
    if index == len(arr):
        return 0                          # Nothing left to sum
    
    # Recursive case: current element + sum of rest
    # Key: index+1 moves toward base case every call
    return arr[index] + sum_array(arr, index + 1)

# Dry run for [1, 2, 3]:
# sum_array([1,2,3], 0) = 1 + sum_array([1,2,3], 1)
#                           = 2 + sum_array([1,2,3], 2)
#                               = 3 + sum_array([1,2,3], 3)
#                                   = 0   (base case)
# Unwind: 3+0=3, 2+3=5, 1+5=6  ✓

sum_array([1,2,3,4,5])

15

Pattern 2: Divide and Conquer

In [3]:
def merge_sort(arr: list[int]) -> list[int]:
    """
    Recurrence: T(n) = 2T(n/2) + O(n)
    Solution:   T(n) = O(n log n)   [Master Theorem, Case 2]
    Space:      O(n) auxiliary + O(log n) stack = O(n)
    """
    # Base case: array of size ≤ 1 is already sorted
    if len(arr) <= 1:
        return arr
    
    # DIVIDE: split into two roughly equal halves
    mid = len(arr) // 2
    left_half = arr[:mid]             # O(n/2) copy — hidden space cost!
    right_half = arr[mid:]            # O(n/2) copy
    
    # CONQUER: recursively sort each half
    sorted_left = merge_sort(left_half)     # T(n/2)
    sorted_right = merge_sort(right_half)   # T(n/2)
    
    # COMBINE: merge two sorted halves → O(n) work
    return _merge(sorted_left, sorted_right)


def _merge(left: list[int], right: list[int]) -> list[int]:
    """Merge two sorted arrays. O(n) time, O(n) space."""
    result = []
    i = j = 0
    
    # Compare front elements, take the smaller one
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    
    # Append any remaining elements
    result.extend(left[i:])
    result.extend(right[j:])
    return result


merge_sort([1,2,3,3,6,7,3,3,7,4,2,8,3,5,2])

[1, 2, 2, 2, 3, 3, 3, 3, 3, 4, 5, 6, 7, 7, 8]

Pattern 3: Multiple Recursion — Generate All Subsets

In [4]:
def generate_subsets(arr: list[int], index: int = 0,
                     current: list[int] = None) -> list[list[int]]:
    """
    At each element, two choices: INCLUDE or EXCLUDE.
    Recurrence: T(n) = 2T(n-1) + O(1)
    Solution:   T(n) = O(2ⁿ)
    
    This exponential complexity is UNAVOIDABLE — there are 2ⁿ subsets.
    """
    if current is None:
        current = []
    
    # Base case: processed all elements → record current subset
    if index == len(arr):
        return [current[:]]         # Return copy of current state
    
    results = []
    
    # Choice 1: EXCLUDE arr[index] — recurse without adding it
    results.extend(generate_subsets(arr, index + 1, current))
    
    # Choice 2: INCLUDE arr[index] — add it, recurse, then remove (backtrack)
    current.append(arr[index])
    results.extend(generate_subsets(arr, index + 1, current))
    current.pop()                   # BACKTRACK — undo the choice
    
    return results

# For arr = [1, 2]:
# Decision tree:
#                    []
#                  /     \
#          exclude 1    include 1
#           []             [1]
#          /  \           /   \
#       excl2 incl2   excl2  incl2
#        []   [2]    [1]    [1,2]
# Output: [[], [2], [1], [1,2]] — all 2² = 4 subsets ✓

generate_subsets([2,3,1])

[[], [1], [3], [3, 1], [2], [2, 1], [2, 3], [2, 3, 1]]

Pattern 4: Tail Recursion vs Head Recursion

In [5]:
# HEAD RECURSION: recursive call comes FIRST
# Work is done AFTER the recursive call returns (on the way back up)
def print_head(n: int) -> None:
    """
    Recursive call first → work done while UNWINDING
    Stack builds up BEFORE any printing
    """
    if n == 0:
        return
    print_head(n - 1)    # Call first
    print(n)             # Work after (prints 1, 2, 3, ... n)

# For n=3: stack builds up [print_head(3), print_head(2), print_head(1), base]
# Then unwinds: prints 1, 2, 3


# TAIL RECURSION: recursive call comes LAST
# Work is done BEFORE the recursive call (on the way down)
def print_tail(n: int) -> None:
    """
    Work first → recursive call last
    No pending work when making the call → theoretically O(1) stack
    (Python doesn't optimize this, but languages like Scheme/Haskell do)
    """
    if n == 0:
        return
    print(n)             # Work first (prints n, n-1, ... 1)
    print_tail(n - 1)    # Call last


# TAIL RECURSIVE FACTORIAL (with accumulator)
def factorial_tail(n: int, accumulator: int = 1) -> int:
    """
    Classic tail recursion pattern: carry the result in an accumulator.
    The recursive call is the VERY LAST operation.
    
    factorial_tail(5, 1)
    → factorial_tail(4, 5)
    → factorial_tail(3, 20)
    → factorial_tail(2, 60)
    → factorial_tail(1, 120)
    → return 120
    
    No "pending multiplication" at any call site.
    In languages with TCO: O(1) stack space.
    In Python: still O(n) stack (Python has no TCO).
    """
    if n <= 1:
        return accumulator               # Return accumulated result
    return factorial_tail(n - 1, n * accumulator)   # Tail call

factorial_tail(5)

120

In [11]:
import sys
sys.set_int_max_str_digits(1420)

In [12]:
def try_by_me(n: int, acc: int = 1):

    if n <= 1:
        return acc
    return try_by_me(n-1, n * acc)

try_by_me(4)



24